# Tutorial 07 - Character Prediction with LSTMs

This tutorial is based on the following two blog posts:
- http://karpathy.github.io/2015/05/21/rnn-effectiveness/
- https://towardsdatascience.com/writing-like-shakespeare-with-machine-learning-in-pytorch-d77f851d910c


The goal of this tutorial is to build a character level RNN and train it on a text corpus of some of Shakespeare's work. We will then use the trained RNN to hallucinate new text (one character at a time). 


![image info](mlp_vs_rnn.png)

![image info](rnn_flow.gif)

![image info](charseq.jpeg)

## Setup and Data Loading

We will use a small amount of Shakespears work to train the character prediction LSTM. Feel free to use different text data to train the network...

In [1]:
# Importing libraries
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F

Open the text file and show the first 100 characters. Feel free to experiment with your own text data of choice!

In [2]:
# Open shakespeare text file and read in data as `text`
with open('shakespeare.txt', 'r') as f:
    text = f.read()
    
# Showing the first 100 characters
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [9]:
len(text)

1115394

## Data Preparation


### 1 - Map every character to one integer
We need to encode the characters in some way. We will simply map each unique character in the text to an integer (and vice versa).

In [26]:
# encoding the text and map each character to an integer and vice versa

# We create two dictionaries:
# 1. int2char, which maps integers to characters
# 2. char2int, which maps characters to integers
chars = tuple(set(text))
int2char = dict(enumerate(chars))
char2int = {ch: ii for ii, ch in int2char.items()}

In [27]:
print(char2int)

{'J': 0, 'z': 1, '\n': 2, 'y': 3, 'O': 4, '?': 5, ';': 6, 'F': 7, "'": 8, 'j': 9, 's': 10, 'x': 11, 'q': 12, 'Q': 13, 'g': 14, 'c': 15, 'a': 16, 'U': 17, 'E': 18, 'T': 19, 'L': 20, 'Y': 21, '-': 22, 'M': 23, 'v': 24, 'n': 25, 'd': 26, ',': 27, 'G': 28, 'r': 29, 'R': 30, 'p': 31, 'D': 32, 'u': 33, 'l': 34, 'S': 35, 'I': 36, 'm': 37, '!': 38, '$': 39, 'N': 40, 'w': 41, 'X': 42, '.': 43, 'k': 44, 'o': 45, 'f': 46, 'A': 47, 'K': 48, 'V': 49, 'P': 50, '&': 51, 'B': 52, 'b': 53, ' ': 54, 't': 55, 'W': 56, 'H': 57, 'Z': 58, 'h': 59, '3': 60, 'i': 61, 'C': 62, ':': 63, 'e': 64}


In [28]:
print('Vocabulary size : ', len(chars))

Vocabulary size :  65


In [11]:
# Encode the text
encoded = np.array([char2int[ch] for ch in text])

# Showing the first 100 encoded characters
encoded[:100]

array([ 7, 61, 29, 10, 55, 54, 62, 61, 55, 61,  1, 64, 25, 63,  2, 52, 64,
       46, 45, 29, 64, 54, 41, 64, 54, 31, 29, 45, 15, 64, 64, 26, 54, 16,
       25,  3, 54, 46, 33, 29, 55, 59, 64, 29, 27, 54, 59, 64, 16, 29, 54,
       37, 64, 54, 10, 31, 64, 16, 44, 43,  2,  2, 47, 34, 34, 63,  2, 35,
       31, 64, 16, 44, 27, 54, 10, 31, 64, 16, 44, 43,  2,  2,  7, 61, 29,
       10, 55, 54, 62, 61, 55, 61,  1, 64, 25, 63,  2, 21, 45, 33])

### 2 - One-hot-encoding

Let's also define some helper functions that map the ineger IDs to one-hot labels. The one-hot encodings will be used as inputs and targets for the network training (similar to multi-class classification).

In [63]:
# Defining method to encode one hot labels
##### arr (batch x seq_length) ----> (batch x seq_length x n_labels)
def one_hot_encode(arr, n_labels):
    ### YOUR CODE HERE ###
    pass

In [64]:
one_hot_encode(arr=torch.LongTensor(np.array([[1, 2, 3, 0]])), n_labels=5)

tensor([[[0., 1., 0., 0., 0.],
         [0., 0., 1., 0., 0.],
         [0., 0., 0., 1., 0.],
         [1., 0., 0., 0., 0.]]])

### 3 - Create a dataset class for training

Finally, define a dataset class that will yield mini-batches for training. 

Suppose that our encoded training text is the following[2, 1, 4, 7, 0, 23, 57, 12]

1- split the text sequence into sub-sequences of lenth "seq_length=4"

[2, 1, 4, 7, 0, 23, 57, 12] ---------> [2, 1, 4, 7], [0, 23, 57, 12]

2- generate input sequence / target sequence for every sub-sequence

First input sequence: [2, 1, 4, 7] ---------> First output sequence: [1, 4, 7, 0] 

Second input sequence: [0, 23, 57, 12] ---------> Second output sequence: [23, 57, 12, 2]



In [87]:
class SubsequencesDataset(Dataset):
    def __init__(self, data: np.ndarray, seq_length: int):
        super(SubsequencesDataset, self).__init__()

        self.data = data
        self.seq_length = seq_length
        
    def __len__(self):
        ### YOUR CODE HERE ###
        pass

    def __getitem__(self, index: int):
        ### YOUR CODE HERE ###
        pass

In [89]:
dataset = SubsequencesDataset(data=np.array([2, 1, 4, 7, 0, 23, 57, 12, 11]), seq_length=4)
len(dataset)

2

In [90]:
dataset[0]

(array([2, 1, 4, 7]), array([1, 4, 7, 0]))

In [91]:
dataset[1]

(array([ 0, 23, 57, 12]), array([23, 57, 12, 11]))

## Define the Recurrent Neural Network Model

First check if a GPU is available for training

In [16]:
# Check if GPU is available
train_on_gpu = torch.cuda.is_available()
if(train_on_gpu):
    print('Training on GPU!')
else: 
    print('No GPU available, training on CPU; consider making n_epochs very small.')

Training on GPU!


Now, it’s time to define the RNN network. We’re going to implement dropout for regularization and also create character dictionaries within the network. We’ll have LSTM units followed by a fully connected layer.

In the forward function, we’ll propagate the input and hidden state values through the LSTM layers to get the output and next hidden state values. After performing dropout, we reshape the output value to make it the proper dimensions for the fully connected layer.

Finally, we also have a section in the RNN for initializing the hidden state values for the correct batch size if using mini-batches.
![image info](lstm.png)

In [17]:
# Declaring the model
class CharRNN(nn.Module):
    
    def __init__(self, tokens, n_hidden=256, n_layers=2, drop_prob=0.5):
        super().__init__()
        
        self.drop_prob = drop_prob
        
        self.n_layers = n_layers
        
        self.n_hidden = n_hidden        
      
        self.chars = tokens
        
        #define the LSTM
        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layers, dropout=drop_prob, batch_first=True)
        
        #define a dropout layer
        self.dropout = nn.Dropout(drop_prob)
        
        #define the final, fully-connected output layer
        self.fc = nn.Linear(n_hidden, len(self.chars))
      
    
    def forward(self, x, hidden):
        ''' Forward pass through the network. 
            These inputs are x, and the hidden/cell state `hidden`. '''
        ### YOUR CODE HERE ###
        pass
    
    
    def init_hidden(self, batch_size):
        ''' Initializes hidden state '''
        # Create two new tensors with sizes n_layers x batch_size x n_hidden,
        # initialized to zero, for hidden state and cell state of LSTM
        weight = next(self.parameters()).data
        
        hidden = (weight.new_zeros(self.n_layers, batch_size, self.n_hidden).zero_(),
                  weight.new_zeros(self.n_layers, batch_size, self.n_hidden).zero_())
        
        return hidden

## Training Code

Time for training! We declare a function, where we define our optimizer (Adam) and the loss function (cross entropy). We then create the training and validation data. 
We loop over the training set, each time encoding the data into one-hot vectors, performing forward and backward passes, and updating the network parameters.

Every once a while, we generate some loss statistics (training loss and validation loss) to let us know if the model is training correctly.

In [100]:
# Declaring the train method
def train(net, data, epochs=10, batch_size=10, seq_length=50, lr=0.001, clip=5, val_frac=0.1, print_every=10):
    ''' Training a network 
    
        Arguments
        ---------
        
        net: CharRNN network
        data: text data to train the network
        epochs: Number of epochs to train
        batch_size: Number of mini-sequences per mini-batch, aka batch size
        seq_length: Number of character steps per mini-batch
        lr: learning rate
        clip: gradient clipping
        val_frac: Fraction of data to hold out for validation
        print_every: Number of steps for printing training and validation loss
    
    '''
    ### YOUR CODE HERE ###
    pass

Now, we just declare the hyperparameters for our model, create an instance for it, and train it!

In [ ]:
# Define and print the net
n_hidden=512
n_layers=2

net = CharRNN(chars, n_hidden, n_layers)
print(net)

# Declaring the hyperparameters
batch_size = 128
seq_length = 100
n_epochs = 20 # start smaller if you are just testing initial behavior

# train the model
train(net, encoded, epochs=n_epochs, batch_size=batch_size, seq_length=seq_length, lr=0.001, print_every=50)

## Generating New Text

After training, we create a method to predict the next character from the trained RNN with forward propagation.

In [104]:
# Defining a method to generate the next character
def predict(net, char, h=None, top_k=None):
        ''' Given a character, predict the next character.
            Returns the predicted character and the hidden state.
            The charachters are sampled from top k if specified.
        '''
        ### YOUR CODE HERE ###
        pass

Then, we define a sampling method that will use the previous method to generate an entire string of text, first using the characters in the first word and then using a loop to generate the next characters.

In [105]:
# Declaring a method to generate new text
def sample(net, size, prime='The', top_k=None):
        
    if(train_on_gpu):
        net.cuda()
    else:
        net.cpu()
    
    net.eval() # eval mode
    
    chars = [ch for ch in prime]
    # Predict next characters here
    # ...
    ### YOUR CODE HERE ###
    
    return ''.join(chars)

Finally, just call the method, define the size you want (e.g., 1000 characters) and the starting character (e.g., ‘A’) and get the result:

In [ ]:
# Generating new text
print(sample(net, 1000, prime='A', top_k=None))